# Milestone 2 : baseline de transport

La **baseline** représente la situation actuelle, avant toute consolidation entre les jours. Elle sert de point de comparaison pour mesurer l'économie apportée par l'outil.

**Règles de construction :**
- une **commande** (ou livraison) correspond à un fournisseur, une usine et un jour ;
- toutes les lignes d'une même commande sont regroupées dans le moins de camions possible, chaque camion restant sous **24,5 t** et **110,4 m³** (SF-07). Par exemple, 4 lots de 5 t donnent 1 camion de 20 t, et non 4 camions de 5 t ;
- **un lot (une ligne du planning) n'est jamais découpé** : si un lot ne rentre dans aucun camion ouvert, il part entier dans un nouveau camion. Un lot de 20 t et un lot de 8 t donnent donc 2 camions de 20 t et 8 t ;
- on part du fichier silver `released_schedule_cleaned.csv`, **complété par les lignes de la quarantaine** (`released_schedule_quarantine.csv`) : ce sont de vrais besoins, qui doivent être livrés (SF-12) :
  - **poids inconnu** (134 lignes) : la ligne ne peut pas être regroupée avec d'autres. Elle a son **propre camion (camion réservé)**, chiffré dans la tranche **> 4 000 kg** (estimation haute) ;
  - **fournisseur sans distance** (4 lignes de SUP_00130, poids connu) : regroupées comme les autres, chiffrées à **> 650 km** (estimation haute) ;
  - **ligne de plus de 24,5 t** (1 ligne, SUP_00074 → PLT_003 le 15/12, 26,9 t) : **exclue pour le moment**, en attente de la réponse du client.

Le résultat est exporté dans `data/baseline/` (camions, commandes et matériaux), en dehors des couches bronze / silver / gold, et toutes les analyses de ce notebook s'appuient sur `baseline_camion.csv`.

La partie 5 estime ensuite le gain que la consolidation peut apporter, et classe les couples fournisseur × usine selon leur potentiel : c'est l'objectif chiffré que le futur moteur de consolidation devra viser.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTINE_PATH = "../data/silver/released_schedule_quarantine.csv"   # lignes mises de côté au milestone 1
BASELINE_DIR = "../data/baseline"
BASELINE_PATH = f"{BASELINE_DIR}/baseline_camion.csv"              # une ligne par camion
BASELINE_COMMANDES_PATH = f"{BASELINE_DIR}/baseline_commande.csv"  # une ligne par commande
BASELINE_MATERIAUX_PATH = f"{BASELINE_DIR}/baseline_materiaux.csv" # une ligne par ligne de besoin

POIDS_MAX = 24500   # kg
VOLUME_MAX = 110.4  # m³

# constantes pour le calcul du coût de transport
FREIGHT_VALUE_MARGIN = 2.3923 # 239,23%
DIESEL_SURCHARGE = 1.1943 # €
DATE_DIESEL_VALIDITY = '2026-08-01'

## Chargement des données de couts : x = poids, y = km
couts = pd.read_csv("../data/bronze/rate_sheet.csv", index_col="km_max")

In [ ]:
# Fonction de calcul du cout de transport en fonction du poids et de la distance (pour la distnace il faut majoré car il n'y a pas tous les km)
# Transport price = Freight cost × (1 + margin) × diesel surcharge

TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]    # 30, 50, ..., 650
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]  # 50, 100, ..., 4000

# Données manquantes : estimations hautes
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1     # poids inconnu -> tranche "> 4 000 kg"
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1  # distance inconnue -> tranche "> 650 km"


def tranche(valeur, tranches):
    """Premiere tranche >= valeur (on majore), sinon la tranche '>max'."""
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"

def cout_transport(poids, distance):
    """Prix d'un envoi en EUR = cout de la grille × (1 + marge) × diesel."""
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))



## 1. Construction de la baseline

### Calcul de la baseline

Une **commande** correspond à un fournisseur, une usine et un jour. Pour chaque commande, les lignes sont triées **du plus lourd au plus léger**, puis chacune est placée dans le **premier camion où elle rentre** (poids et volume). Si elle ne rentre dans aucun camion déjà ouvert, elle ouvre un nouveau camion. Les lignes plus légères qui suivent peuvent encore compléter les premiers camions : c'est l'heuristique *First Fit Decreasing*, qui remplit au maximum le premier camion.

Une ligne au **poids inconnu** ne peut pas être placée : elle part seule dans un **camion réservé**, que les autres lignes ne peuvent pas compléter.

Chaque commande reçoit un `commande_id` et chaque camion un `camion_id` uniques. Les colonnes `source` (silver ou quarantaine) et `reserve` (camion réservé) permettent de retrouver les lignes de la quarantaine. Trois tables sont exportées dans `data/baseline/` (les fichiers existants sont écrasés) :
- `baseline_camion.csv` : une ligne par camion (poids et volume chargés) ;
- `baseline_commande.csv` : une ligne par commande (nombre de camions, de lignes, poids et volume totaux) ;
- `baseline_materiaux.csv` : une ligne par ligne de besoin, avec sa commande et son camion.

Les contrôles vérifient le résultat avant l'export : aucun poids n'est perdu, aucun camion ne dépasse 24,5 t ni 110,4 m³, chaque camion réservé ne porte qu'une ligne, et chaque ligne (silver et quarantaine) est affectée à un et un seul camion (SF-12).

In [ ]:
silver_schedule = pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"])
silver_schedule["source"] = "silver"
silver_schedule["ligne_id"] = silver_schedule.index  # n° de la ligne dans son fichier (traçabilité)

# Lignes de la quarantaine : de vrais besoins, à livrer eux aussi (SF-12)
quarantaine = pd.read_csv(QUARANTINE_PATH, parse_dates=["need_date"])
quarantaine["source"] = "quarantaine"
quarantaine["ligne_id"] = quarantaine.index
trop_lourd = quarantaine["weight"] > POIDS_MAX   # ligne > 24,5 t : exclue, en attente de la réponse du client
schedule = pd.concat([silver_schedule, quarantaine[~trop_lourd].drop(columns="flags")], ignore_index=True)
schedule["reserve"] = schedule["weight"].isna()   # poids inconnu : un camion réservé pour la ligne

KEYS = ["need_date", "supplier_id", "plant"]  # une commande = un fournisseur, une usine, un jour

# Tri : commande, puis poids du plus lourd au plus léger (les poids inconnus à la fin)
df = schedule.sort_values(KEYS + ["weight"], ascending=[True, True, True, False], na_position="last").copy()

numeros_camion = []
for _, lignes in df.groupby(KEYS, sort=False):
    camions_ouverts = []  # [poids, volume] déjà chargés dans chaque camion de la commande
    for poids_ligne, volume_ligne in zip(lignes["weight"], lignes["volume"]):
        if pd.isna(poids_ligne):  # poids inconnu : camion réservé à cette ligne, fermé aux autres
            camions_ouverts.append([np.inf, np.inf])
            numeros_camion.append(len(camions_ouverts))
            continue
        for i, (poids_camion, volume_camion) in enumerate(camions_ouverts):
            if poids_camion + poids_ligne <= POIDS_MAX and volume_camion + volume_ligne <= VOLUME_MAX:
                camions_ouverts[i] = [poids_camion + poids_ligne, volume_camion + volume_ligne]
                numeros_camion.append(i + 1)
                break
        else:  # la ligne ne rentre dans aucun camion ouvert -> nouveau camion
            camions_ouverts.append([poids_ligne, volume_ligne])
            numeros_camion.append(len(camions_ouverts))

df["camion"] = numeros_camion                                     # n° du camion dans la commande (1, 2, ...)
df["commande_id"] = df.groupby(KEYS).ngroup() + 1                 # id unique de commande
df["camion_id"] = df.groupby(KEYS + ["camion"]).ngroup() + 1      # id unique de camion
df = df.rename(columns={"need_date": "date"})
somme = lambda s: s.sum(min_count=1)                              # NaN si aucune valeur connue

# Table 1 : une ligne par camion
baseline_camion = (df.groupby(["camion_id", "commande_id", "supplier_id", "plant", "date", "camion"])
                     .agg(poids=("weight", somme), volume=("volume", somme), nb_lignes=("weight", "size"),
                          reserve=("reserve", "any"),
                          source=("source", lambda s: "quarantaine" if s.eq("quarantaine").any() else "silver"))
                     .reset_index())

# Table 2 : une ligne par commande
baseline_commande = (df.groupby(["commande_id", "supplier_id", "plant", "date"])
                       .agg(nb_camions=("camion", "max"), nb_camions_reserves=("reserve", "sum"),
                            nb_lignes=("camion", "size"), poids=("weight", "sum"), volume=("volume", "sum"))
                       .reset_index())

# Table 3 : une ligne par ligne de besoin, avec sa commande et son camion
baseline_materiaux = (df[["source", "ligne_id", "commande_id", "camion_id", "supplier_id", "plant", "date",
                          "material", "unloading_point", "schedule_quantity", "weight", "volume", "reserve"]]
                        .sort_values(["commande_id", "camion_id", "weight"], ascending=[True, True, False]))

# Contrôles
connus = ~baseline_camion["reserve"]
assert abs(baseline_camion["poids"].sum() - schedule["weight"].sum()) < 1e-6           # aucun poids perdu
assert (baseline_camion.loc[connus, "poids"] <= POIDS_MAX).all()                         # SF-07 poids
assert (baseline_camion.loc[connus, "volume"] <= VOLUME_MAX).all()                       # SF-07 volume
assert (baseline_camion.loc[~connus, "nb_lignes"] == 1).all()                            # un camion réservé = une ligne
cle = ["source", "ligne_id"]
assert not baseline_materiaux.duplicated(cle).any()                                      # SF-12 pas de doublon
assert set(map(tuple, baseline_materiaux[cle].values)) == set(map(tuple, schedule[cle].values))  # SF-12 pas d'oubli

print(f"Lignes : {len(silver_schedule)} du silver + {(~trop_lourd).sum()} de la quarantaine "
      f"({schedule['reserve'].sum()} sans poids, {(~trop_lourd).sum() - schedule['reserve'].sum()} avec un poids connu)")
print(f"Exclue en attente de la réponse du client : {trop_lourd.sum()} ligne de plus de 24,5 t "
      f"({', '.join(f'{r.supplier_id} -> {r.plant}, {r.need_date:%d/%m}, {r.weight / 1000:.1f} t'.replace('.', ',') for r in quarantaine[trop_lourd].itertuples())})")
print(f"Camions : {connus.sum()} consolidés + {(~connus).sum()} réservés (poids inconnu) = {len(baseline_camion)}")

# Export (le dossier data/baseline/ est créé s'il n'existe pas, les fichiers existants sont écrasés)
os.makedirs(BASELINE_DIR, exist_ok=True)
for table, path in [(baseline_camion, BASELINE_PATH),
                    (baseline_commande, BASELINE_COMMANDES_PATH),
                    (baseline_materiaux, BASELINE_MATERIAUX_PATH)]:
    # arrondi à 6 décimales : enlève le bruit des sommes en virgule flottante (521.9000000000001 -> 521.9),
    # sinon le CSV change selon la machine. Sans perte : silver <= 3 décimales (poids), <= 6 (volume)
    table.round(dict.fromkeys(table.select_dtypes("float").columns, 6)).to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"Exporté : {path} ({len(table)} lignes)")

### Chargement de la baseline

Toutes les cellules suivantes travaillent à partir du fichier `baseline_camion.csv` (une ligne par camion).

In [ ]:
# Une ligne par camion
camions = pd.read_csv(BASELINE_PATH, parse_dates=["date"])
camions["remplissage_poids"] = camions["poids"] / POIDS_MAX
camions["remplissage_volume"] = camions["volume"] / VOLUME_MAX

# Une ligne par livraison (fournisseur × usine × jour) = une commande de la baseline
livraisons = pd.read_csv(BASELINE_COMMANDES_PATH, parse_dates=["date"])
livraisons["semaine"] = livraisons["date"].dt.isocalendar().week
livraisons["jour"] = livraisons["date"].dt.day_of_week  # 0 = lundi
livraisons["remplissage_poids"] = livraisons["poids"] / (livraisons["nb_camions"] * POIDS_MAX)


# Camions dont le poids est connu : les seuls utilisables pour analyser le remplissage
camions_connus = camions[~camions["reserve"]]

print(f"{len(livraisons)} livraisons, {len(camions)} camions dont {camions['reserve'].sum()} réservés (poids inconnu)")
print(f"Poids max d'un camion : {camions['poids'].max():.1f} kg")

#### Analyse : livraisons et camions

La baseline compte **1 665 livraisons** et **1 741 camions** : 1 607 camions consolidés et **134 camions réservés** pour les lignes sans poids. Sur les seules lignes du silver, on retrouve 1 599 livraisons et 1 603 camions.

Les camions réservés expliquent presque tous les écarts entre livraisons et camions : 121 livraisons contiennent au moins une ligne sans poids, dont 62 qui n'ont que des lignes sans poids. Hors camions réservés, seules 4 livraisons demandent deux camions, parce que leur poids total dépasse 24,5 t :

| Livraison | Poids total | Camions |
|---|---|---|
| SUP_00046 → PLT_004, 11/11 | 46,7 t | 2 |
| SUP_00046 → PLT_004, 02/12 | 32,3 t | 2 |
| SUP_00046 → PLT_004, 25/11 | 26,5 t | 2 |
| SUP_00037 → PLT_002, 27/11 | 25,7 t | 2 |

*Remarque : grâce au tri des lots du plus lourd au plus léger, la livraison du 11/11 (46,7 t) tient dans 2 camions, soit le minimum possible (46,7 / 24,5 = 1,9). Une répartition dans l'ordre du fichier en utilisait 3.*

---

## 2. Répartition des livraisons dans le temps

### Livraisons par jour de la semaine

On regarde comment les livraisons se répartissent entre les jours de la semaine.

In [ ]:
jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]

def stats_par_jour(camions, date_col="date"):
    """Par date : nb de livraisons (couple supplier x plant) et nb de camions,
    puis moyenne par jour de la semaine."""
    df = camions.assign(date=pd.to_datetime(camions[date_col]))
    par_date = pd.DataFrame({
        "nb_livraisons": df.drop_duplicates(["date", "supplier_id", "plant"]).groupby("date").size(),
        "nb_camions": df.groupby("date").size(),
    })
    par_date["jour_semaine"] = par_date.index.day_of_week
    moyenne = (par_date.groupby("jour_semaine")[["nb_livraisons", "nb_camions"]]
               .mean().reindex(range(5)))
    moyenne.index = jours
    return par_date, moyenne


par_date_base, moy_base = stats_par_jour(camions)
sem_base = par_date_base.groupby(par_date_base.index.to_period("W-SUN").start_time)["nb_camions"].sum()

print(moy_base.round(1))
print(f"Nombre moyen de camions par semaine : {sem_base.mean():.1f}")

scenarios = {
    "Baseline": (moy_base, sem_base),
    # "Après optimisation": (moy_opti, sem_opti),
}

fig, ax = plt.subplots(figsize=(9, 5))

# barres : livraisons moyennes de la baseline
bars = ax.bar(jours, moy_base["nb_livraisons"], color="#9ec5d6",
              label="Livraisons moyennes (couple × jour)")
ax.bar_label(bars, fmt="%.1f", padding=3)

# traits pointillés : camions moyens par scénario
for nom, (moy, sem) in scenarios.items():
    ax.plot(jours, moy["nb_camions"], linestyle="--", marker="o",
            label=f"Camions moyens – {nom} ({sem.mean():.0f} camions / sem.)")

ax.set_title("Livraisons et camions moyens par jour de la semaine")
ax.set_xlabel("Jour")
ax.set_ylabel("Nombre moyen")
ax.set_ylim(0, moy_base.values.max() * 1.2)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


#### Analyse : livraisons par jour de la semaine

Les livraisons n'ont lieu qu'en semaine, avec en moyenne **290 camions par semaine** sur les 6 semaines du planning (de 173 à 402 selon la semaine).

**Les deux courbes sont proches** : dans la baseline, chaque livraison part dans son propre camion. L'écart d'environ 2 à 3 camions par jour vient surtout des **camions réservés** : une livraison qui contient une ligne sans poids demande un camion de plus. C'est précisément ce que la consolidation doit changer : après optimisation, la courbe des camions devra passer nettement **sous** les barres des livraisons.

La charge est plus forte en fin de semaine : le **jeudi** est le jour le plus chargé (environ 65 camions) et le **mercredi** le plus calme (environ 52). Lors de la consolidation, il faudra veiller à ne pas accentuer ce déséquilibre en répartissant les jours de livraison entre les couples d'une même usine (SF-08).

---

### Nombre de camions par semaine

In [ ]:
# Nombre de camions par semaine, au total et par usine
camions["semaine"] = camions["date"].dt.isocalendar().week

par_semaine = camions.groupby("semaine").size()
par_semaine_usine = (camions.pivot_table(index="semaine", columns="plant", values="camion", aggfunc="size")
                            .fillna(0).astype(int))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(par_semaine.index, par_semaine.values, marker="o", linewidth=2.5, color="black", label="Total")
for usine in par_semaine_usine.columns:
    ax.plot(par_semaine_usine.index, par_semaine_usine[usine], marker="o", label=usine)

for x, y in par_semaine.items():  # valeur au-dessus de chaque point du total
    ax.annotate(y, (x, y), textcoords="offset points", xytext=(0, 6), ha="center")

ax.set_xticks(par_semaine.index, [f"S{s}" for s in par_semaine.index])
ax.set_ylim(bottom=0)
ax.set_title("Nombre de camions par semaine (baseline)")
ax.set_xlabel("Semaine")
ax.set_ylabel("Nombre de camions")
ax.grid(alpha=0.4)
ax.legend()
plt.show()

par_semaine_usine.assign(Total=par_semaine)


#### Analyse : camions par semaine

Sur les 6 semaines du planning, la baseline compte 1 741 camions (dont 134 réservés), soit environ 290 camions par semaine en moyenne, ou près de 60 par jour ouvré.

**Une charge irrégulière**
- S46-S47 : activité stable, autour de 308 camions par semaine.
- S48 : pic à 402 camions. Il vient presque entièrement de PLT_002, qui double, ce qui correspond au pic des 26 et 27/11 relevé dans le milestone 1.
- S49 à S51 : baisse continue jusqu'à 173 camions en S51, avec l'approche de la fermeture de fin d'année. PLT_002 n'a plus aucun camion en S51.

**Des profils très différents selon l'usine**
| Usine | Camions par semaine | Comportement |
|---|---|---|
| PLT_004 | 88 à 97 | Très stable, y compris en décembre |
| PLT_001 | 57 à 110 | Stable, puis baisse en décembre |
| PLT_002 | 0 à 163 | Très irrégulière : pic en S48, arrêt en S51 |
| PLT_003 | 24 à 37 | Faible volume, assez régulier |

On en déduit que : 
- PLT_004 est la cible prioritaire : sa charge est régulière et ses couples livrent presque tous les jours (voir la carte de chaleur). Un profil fixe, par exemple 1 ou 2 livraisons par semaine, y sera valable toutes les semaines.
- Cette courbe servira de référence : la même courbe tracée après consolidation montrera, semaine par semaine, le nombre de camions économisés.
- PLT_002 sera sans doute la courbe qui absorbera le mieux l'optimisation.

---

### Fréquence de livraison des couples les plus actifs

La carte de chaleur montre, pour les 10 couples fournisseur × usine qui livrent le plus souvent, le nombre de livraisons par jour de la semaine sur l'horizon du planning.

In [ ]:
# extraction des livraisons uniques et on définit le couple et le jour
df_livr = livraisons.drop_duplicates(['date', 'supplier_id', 'plant']).copy()
df_livr['couple'] = df_livr['supplier_id'] + " × " + df_livr['plant']
jours_fr = {0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi', 4: 'Vendredi'}
df_livr['jour'] = pd.to_datetime(df_livr['date']).dt.day_of_week.map(jours_fr)

# nombre de passages pour chaque couple selon le jour
jours = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi']
table = pd.crosstab(df_livr['couple'], df_livr['jour'])[jours]

# top 10
top_10 = table.sum(axis=1).nlargest(10).index
heat = table.loc[top_10]

# carte de chaleur avec matplotlib seul
fig, ax = plt.subplots(figsize=(8, 10))
# échelle forcée de 0 à 6 (6 semaines = maximum possible) : en automatique elle allait de 5 à 6,
# et un 5 apparaissait blanc comme une case vide
im = ax.imshow(heat.values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=6)
cbar = fig.colorbar(im, ax=ax, ticks=range(7))
cbar.set_label("Nombre de livraisons sur les 6 semaines (6 = livre chaque semaine ce jour-là)")

ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_yticks(range(len(heat.index)), heat.index)

# valeur écrite dans chaque case (en blanc sur les cases foncées)
seuil = heat.values.max() / 2
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        valeur = heat.values[i, j]
        ax.text(j, i, valeur, ha="center", va="center",
                color="white" if valeur > seuil else "black")

ax.set_title("Carte de chaleur : Top 10 des couples × Jours de livraison")
ax.set_xlabel("Jour de la semaine")
ax.set_ylabel("Couple (Fournisseur × Usine)")
plt.show()

#### Analyse : carte de chaleur

Les couples les plus actifs livrent **tous les jours de la semaine** : 5 à 6 livraisons par jour de semaine, soit environ une livraison par jour ouvré sur les 6 semaines du planning. La plupart livrent l'usine **PLT_004**.

Ce sont les meilleurs candidats à la consolidation : passer d'une livraison par jour à 1 ou 2 livraisons par semaine (SF-04, SF-05), en livrant certains besoins en avance (SF-06), réduirait fortement leur nombre de camions sans dépasser la capacité, vu le faible remplissage actuel.

---

## 3. Remplissage des camions

On regarde le nombre de camions par livraison, puis le poids, le taux de remplissage et le volume de chaque camion. Les **134 camions réservés** (poids inconnu) sont exclus de l'analyse du poids et du volume : on ne connaît pas leur chargement.

In [ ]:
# ---------- Nombre de camions par livraison (tableau, pas de graphique) ----------
print("Nombre de livraisons selon leur nombre de camions :")
print(livraisons["nb_camions"].value_counts().sort_index())
print(livraisons[livraisons["nb_camions"] - livraisons["nb_camions_reserves"] > 1])   # trop lourdes pour un camion

### Poids par camion

Deux graphiques côte à côte pour le poids : à gauche les camions sous 4 000 kg (dernière tranche de la grille tarifaire), par intervalles de 250 kg ; à droite tous les camions sur l'échelle complète, de 0 à 24,5 t. Vient ensuite le taux de remplissage en poids (poids / 24 500 kg).

In [ ]:
# ---------- Poids par camion ----------
def nombre(x):
    """24500 -> '24 500', 110.4 -> '110,4' (format français)."""
    return f"{x:,.1f}".replace(",", " ").replace(".", ",").removesuffix(",0")


def pourcent(x):
    """0.956 -> '95,6 %'."""
    return f"{x:.1%}".replace(".", ",").replace("%", " %")


def histo_zoom_et_complet(serie, unite, couleur, seuil, pas, capacite, legende_seuil):
    """Deux histogrammes côte à côte, à intervalles réguliers :
    - à gauche, les camions sous le seuil, par intervalles de `pas` ;
    - à droite, tous les camions sur l'échelle complète (0 à la capacité), par intervalles de 10 % de la capacité."""
    sous = serie <= seuil
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))

    ax1.hist(serie[sous], bins=np.arange(0, seuil + pas, pas), color=couleur, edgecolor="black")
    ax1.axvline(serie.median(), color="black", linestyle="--", label=f"médiane : {nombre(serie.median())} {unite}")
    ax1.set_xlim(0, seuil)
    ax1.set_title(f"Sous {nombre(seuil)} {unite} : {sous.sum()} camions sur {len(serie)} ({pourcent(sous.mean())})\n"
                  f"intervalles de {nombre(pas)} {unite}")

    _, _, barres = ax2.hist(serie, bins=np.linspace(0, capacite, 11), color=couleur, edgecolor="black")
    ax2.bar_label(barres)
    ax2.axvline(seuil, color="red", linestyle="--", label=legende_seuil)
    ax2.set_xlim(0, capacite)
    ax2.margins(y=0.1)
    ax2.set_title(f"Échelle complète (0 à {nombre(capacite)} {unite}) : {(~sous).sum()} camions au-delà de "
                  f"{nombre(seuil)} {unite}\nmax {nombre(serie.max())} {unite} ({pourcent(serie.max() / capacite)} de la capacité), "
                  f"intervalles de 10 % de la capacité")

    for ax in (ax1, ax2):
        ax.set_xlabel(f"{serie.name.capitalize()} ({unite})")
        ax.set_ylabel("Nombre de camions")
        ax.grid(axis="y", alpha=0.5)
        ax.set_axisbelow(True)
        ax.legend()
    fig.suptitle(f"{serie.name.capitalize()} par camion", fontsize=14)
    plt.tight_layout()
    plt.show()

    print(f"{serie.name.capitalize()} médian : {nombre(serie.median())} {unite}")
    print(f"Camions sous {nombre(seuil)} {unite} : {sous.sum()} sur {len(serie)} ({pourcent(sous.mean())})")
    print(f"Camions au-delà de {nombre(seuil)} {unite} : {(~sous).sum()} ({pourcent((~sous).mean())})")
    print(f"Maximum : {nombre(serie.max())} {unite} ({pourcent(serie.max() / capacite)} de la capacité)")


# 4 000 kg = dernière tranche de la grille tarifaire : au-delà, le prix d'un camion est quasi fixe
histo_zoom_et_complet(camions_connus["poids"], "kg", "royalblue", seuil=TRANCHES_KG[-1], pas=250, capacite=POIDS_MAX,
                      legende_seuil=f"{nombre(TRANCHES_KG[-1])} kg : fin de la grille tarifaire")
print(f"Camions sous 500 kg : {pourcent((camions_connus['poids'] <= 500).mean())}")
print(f"Camions au-delà de 20 t : {(camions_connus['poids'] > 20000).sum()}")
print("Poids médian par usine (kg) :", camions_connus.groupby("plant")["poids"].median().round(0).to_dict())


# ---------- Taux de remplissage en poids par camion ----------
# Par tranches : avec 30 barres de 0 à 100 %, tout serait dans la première
classes = pd.cut(camions_connus["remplissage_poids"] * 100, bins=[0, 1, 2, 5, 10, 25, 50, 75, 100],
                 include_lowest=True).value_counts().sort_index()
classes.index = ["0-1 %", "1-2 %", "2-5 %", "5-10 %", "10-25 %", "25-50 %", "50-75 %", "75-100 %"]
ax = classes.plot(kind="bar", color="darkorange", edgecolor="black", figsize=(8, 4), rot=0)
ax.bar_label(ax.containers[0])
plt.title(f"Taux de remplissage en poids des camions (médiane : {camions_connus['remplissage_poids'].median():.1%})")
plt.xlabel("Taux de remplissage")
plt.ylabel("Nombre de camions")
plt.grid(axis="y", alpha=0.5)
plt.show()

taux = camions_connus["remplissage_poids"]
print(f"Taux de remplissage médian en poids : {pourcent(taux.median())}")
print(f"Camions remplis à moins de 1 % : {(taux <= 0.01).sum()} ({pourcent((taux <= 0.01).mean())})")
print(f"Camions remplis à moins de 5 % : {(taux <= 0.05).sum()} ({pourcent((taux <= 0.05).mean())})")
print(f"Camions remplis à plus de 50 % : {(taux > 0.5).sum()}")

#### Analyse : poids et taux de remplissage en poids

Sur les 1 607 camions dont le poids est connu, le poids médian est de **323 kg**, soit un taux de remplissage médian de **1,3 %** : les camions sont très peu remplis. **95,6 % des camions (1 536 sur 1 607) pèsent moins de 4 000 kg**, et 63,7 % moins de 500 kg. Sur l'échelle complète, presque tous les camions tombent dans le premier intervalle (0 à 10 % de la capacité). Le taux de remplissage le confirme : 708 camions (44,1 %) sont remplis à moins de 1 %, 84,1 % à moins de 5 %, et seuls 21 dépassent 50 %. À l'autre extrême, 14 camions dépassent 20 t : ce sont surtout les livraisons trop lourdes pour un seul camion.

Or la grille tarifaire est dégressive : le prix au kilo baisse avec le poids, et au-delà de 4 000 kg le prix d'un camion ne bouge presque plus jusqu'à 24,5 t. L'optimisation doit donc regrouper les envois pour rapprocher les camions de 24,5 t, et au minimum leur faire dépasser 4 000 kg, là où le kilo transporté coûte le moins cher.

---

### Volume par camion

Même présentation que pour le poids : à gauche les camions sous 10 m³, à droite l'échelle complète jusqu'à 110,4 m³, puis le taux de remplissage en volume (volume / 110,4 m³).

In [ ]:
# ---------- Volume par camion ----------
histo_zoom_et_complet(camions_connus["volume"], "m³", "forestgreen", seuil=10, pas=0.5, capacite=VOLUME_MAX,
                      legende_seuil="10 m³")

# ---------- Taux de remplissage en volume par camion ----------
# Par tranches : avec 30 barres de 0 à 100 %, tout serait dans la première
classes = pd.cut(camions_connus["remplissage_volume"] * 100, bins=[0, 1, 2, 5, 10, 25, 50, 75, 100],
                 include_lowest=True).value_counts().sort_index()
classes.index = ["0-1 %", "1-2 %", "2-5 %", "5-10 %", "10-25 %", "25-50 %", "50-75 %", "75-100 %"]
ax = classes.plot(kind="bar", color="purple", edgecolor="black", figsize=(8, 4), rot=0)
ax.bar_label(ax.containers[0])
plt.title(f"Taux de remplissage en volume des camions (médiane : {camions_connus['remplissage_volume'].median():.1%})")
plt.xlabel("Taux de remplissage")
plt.ylabel("Nombre de camions")
plt.grid(axis="y", alpha=0.5)
plt.show()

taux = camions_connus["remplissage_volume"]
print(f"Taux de remplissage médian en volume : {pourcent(taux.median())}")
print(f"Taux de remplissage maximal en volume : {pourcent(taux.max())}")
print(f"Camions remplis à plus de 50 % en volume : {(taux > 0.5).sum()}")

#### Analyse : volume et taux de remplissage en volume

Le volume médian d'un camion est de **1,2 m³**, soit un taux de remplissage médian de **1,1 %**. **99,1 % des camions dont le poids est connu transportent moins de 10 m³**, et le plus volumineux occupe 36,6 m³, soit **33,2 % de la capacité** : aucun camion n'utilise ne serait-ce que la moitié du volume disponible. Sur l'échelle complète, seuls 14 camions dépassent le premier intervalle.

Le taux de remplissage en volume est du même ordre que celui en poids : même les camions les plus lourds gardent beaucoup de place. Le volume ne sera donc pas la contrainte qui limitera la consolidation.

---

### Poids × volume

In [ ]:
# ---------- Poids × volume par camion ----------
camions_connus.plot(kind="scatter", x="poids", y="volume", alpha=0.4, color="purple", figsize=(8, 5))
plt.axvline(POIDS_MAX, color="red", linestyle="--", label="24,5 t")
plt.axhline(VOLUME_MAX, color="orange", linestyle="--", label="110,4 m³")
plt.xlim(0, POIDS_MAX * 1.05)
plt.ylim(0, VOLUME_MAX * 1.05)
plt.title("Poids × volume par camion")
plt.xlabel("Poids (kg)")
plt.ylabel("Volume (m³)")
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()

#### Analyse : poids × volume

Le nuage de points est concentré en bas à gauche : la plupart des camions roulent presque à vide, en poids comme en volume. Les quelques camions lourds atteignent 24,5 t sans dépasser 37 m³ : **c'est le poids, et non le volume, qui limitera la consolidation.**

---

### Synthèse du remplissage

- **Nombre de camions** : hors camions réservés, seules 4 livraisons sur 1 665 demandent plus d'un camion. Les 134 lignes sans poids ajoutent chacune un camion réservé.
- **Poids** (camions dont le poids est connu) : le poids médian d'un camion est d'environ **323 kg** pour une capacité de 24 500 kg. 95,6 % des camions pèsent moins de 4 000 kg, mais 14 dépassent 20 t.
- **Taux de remplissage** : la médiane est de **1,3 %**. 708 camions sont remplis à moins de 1 %, et 84,1 % le sont à moins de 5 %. Seuls 21 camions dépassent 50 %.
- **Par usine** : le remplissage reste très faible partout. PLT_003 a les camions les plus chargés (médiane d'environ 580 kg), PLT_001 les moins chargés (environ 240 kg).
- **Volume** : le volume médian est d'environ 1,2 m³ et le maximum de 36,6 m³ (33,2 % de la capacité), très loin de la limite de 110,4 m³. C'est le **poids** qui limitera la consolidation, pas le volume.

## 4. Coût de transport

### Coût par usine et par semaine

Chaque camion de la baseline est chiffré avec `cout_transport(poids, distance)` : la distance du fournisseur vient de `distances.csv`, le poids est celui du camion. Le prix suit la grille tarifaire (tranches de km et de kg, majorées), multipliée par la marge (× (1 + 2,3923), soit +239,23 %) et la surcharge diesel (× 1,1943). Deux estimations hautes pour les données manquantes : un camion réservé (poids inconnu) est chiffré dans la tranche **> 4 000 kg**, et un fournisseur sans distance à **> 650 km**. On additionne ensuite les coûts par usine et par semaine.

In [ ]:
# Distance de chaque fournisseur (le fichier a un BOM -> utf-8-sig)
distances = pd.read_csv("../data/bronze/distances.csv", encoding="utf-8-sig")
camions["distance"] = camions["supplier_id"].map(distances.set_index("supplier_id")["distance"])
camions["distance_estimee"] = camions["distance"].isna()            # fournisseur sans distance
camions["distance"] = camions["distance"].fillna(DISTANCE_PAR_DEFAUT)  # estimation haute : > 650 km

# Coût de chaque camion (poids inconnu -> tranche > 4 000 kg, estimation haute)
camions["cout"] = [cout_transport(POIDS_PAR_DEFAUT if r else p, d)
                   for p, d, r in zip(camions["poids"], camions["distance"], camions["reserve"])]
print(f"Camions réservés (poids inconnu) : {camions['reserve'].sum()}, {camions.loc[camions['reserve'], 'cout'].sum():,.0f} €"
      .replace(",", " "))
print(f"Camions de fournisseurs sans distance : {camions['distance_estimee'].sum()}, "
      f"{camions.loc[camions['distance_estimee'], 'cout'].sum():,.0f} €".replace(",", " "))
print(f"Lignes du silver seules : {(camions['source'] == 'silver').sum()} camions, "
      f"{camions.loc[camions['source'] == 'silver', 'cout'].sum():,.0f} €".replace(",", " "))
camions["semaine"] = camions["date"].dt.isocalendar().week

# Coût total par usine et par semaine
cout_semaine_usine = (camions.pivot_table(index="semaine", columns="plant", values="cout", aggfunc="sum")
                             .fillna(0))
cout_semaine = cout_semaine_usine.sum(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
cout_semaine_usine.plot(kind="bar", stacked=True, ax=ax, rot=0, edgecolor="black")
for x, y in enumerate(cout_semaine):  # total au-dessus de chaque barre
    ax.annotate(f"{y:,.0f} €".replace(",", " "), (x, y), textcoords="offset points", xytext=(0, 4), ha="center")

ax.set_xticks(range(len(cout_semaine)), [f"S{s}" for s in cout_semaine.index])
ax.set_ylim(0, cout_semaine.max() * 1.12)
ax.set_title(f"Coût de transport par semaine et par usine (baseline, total {camions['cout'].sum():,.0f} €)"
             .replace(",", " "))
ax.set_xlabel("Semaine")
ax.set_ylabel("Coût (€)")
ax.grid(axis="y", alpha=0.4)
ax.set_axisbelow(True)
ax.legend(title="Usine")
plt.show()

# Tableau récapitulatif (€)
recap = cout_semaine_usine.assign(Total=cout_semaine)
recap.loc["Total"] = recap.sum()
recap.round(0)

#### Analyse : coût par semaine

Le coût total de la baseline est de **177 321 €** sur les 6 semaines. Il suit la courbe du nombre de camions : pic en S48 (39 899 €), puis baisse en décembre jusqu'à 17 782 € en S51. PLT_004 est l'usine la plus chère (58 588 €), devant PLT_002 (45 491 €) et PLT_001 (45 310 €).

**La qualité des données pèse lourd** : les 134 camions réservés (lignes sans poids) coûtent à eux seuls **33 506 €, soit 18,9 % du total**, en estimation haute. Les 4 camions du fournisseur sans distance (SUP_00130) coûtent 850 €. Sur les seules lignes du silver, le coût est de 142 965 €. Ce surcoût ne dépend pas de la consolidation : il ne baissera que si les poids manquants sont renseignés.

Après optimisation, le coût devrait baisser avec le nombre de camions : ce graphique servira de point de comparaison, semaine par semaine.

---

### Les 10 couples les plus chers

In [ ]:
# 10 couples les plus chers couple fournisseur x usine
# Le coût de chaque camion (tarif de son propre poids, chaque camion est un envoi facturé) est déjà
# calculé dans camions["cout"] (cellule "Coût par usine et par semaine") : on ne recharge pas le CSV,
# sinon camions est écrasé et perd ses colonnes distance et cout pour les cellules suivantes.

# Une ligne par couple fournisseur × usine
couples = (camions.groupby(["supplier_id", "plant"])
              .agg(nb_commandes=("commande_id", "nunique"),   # nunique : une commande peut avoir 2 camions
                   nb_camions=("camion_id", "size"),
                   poids=("poids", "sum"),
                   cout=("cout", "sum"))
              .reset_index())
couples["couple"] = couples["supplier_id"] + " × " + couples["plant"]

cout_total_txt = f"{couples['cout'].sum():,.2f}".replace(",", " ").replace(".", ",")  # 142 964,96
print(f"{len(couples)} couples, coût total : {cout_total_txt} €")

# ---------- Histogramme horizontal : les 10 couples les plus chers ----------
top = couples.nlargest(10, "cout").sort_values("cout")      # tri croissant : le plus cher en haut
ax = top.plot(kind="barh", x="couple", y="cout", legend=False,
              color="royalblue", edgecolor="black", figsize=(8, 6))
ax.bar_label(ax.containers[0], fmt="%.0f €")
plt.title("Coût de transport par couple fournisseur × usine (10 plus élevés)")
plt.xlabel("Coût total sur la période (€)")
plt.ylabel("")
plt.grid(axis="x", alpha=0.5)
plt.show()

In [ ]:
# ---------- Part des 10 couples les plus chers dans le coût total ----------
cout_total = couples["cout"].sum()
top10 = couples.nlargest(10, "cout")
cout_top10 = top10["cout"].sum()
part = cout_top10 / cout_total * 100

print(f"Les 10 couples les plus chers : {cout_top10:,.2f} € sur {cout_total:,.2f} € "
      f"= {part:.1f} % du coût total".replace(",", " ").replace(".", ","))
print(f"Ils représentent {10 / len(couples) * 100:.1f} % des couples "
      f"({10} sur {len(couples)})".replace(".", ","))

# Part de chaque couple dans le coût total
top10 = top10.assign(part_cout=top10["cout"] / cout_total * 100)
print(top10[["couple", "nb_camions", "cout", "part_cout"]]
      .round({"cout": 2, "part_cout": 1}).to_string(index=False))

#### Analyse : couples les plus chers

Sur 180 couples, les 10 plus chers représentent **27,5 % du coût total**. **4 d'entre eux doivent leur place surtout aux camions réservés** :

| Couple | Camions | dont réservés | Coût | dont camions réservés |
|---|---|---|---|---|
| SUP_00129 → PLT_003 | 50 | 24 | 10 144 € | 7 102 € |
| SUP_00129 → PLT_001 | 24 | 17 | 5 938 € | 5 031 € |
| SUP_00001 → PLT_004 | 18 | 16 | 3 593 € | 3 423 € |
| SUP_00153 → PLT_004 | 11 | 11 | 3 255 € | 3 255 € |

Pour ces couples, l'économie passe d'abord par la qualité des données : tant que les poids manquent, la consolidation ne peut rien faire.

Parmi les couples dont les données sont complètes, **SUP_00085 → PLT_004** arrive en tête (7 429 €), et PLT_004 reste l'usine dont les fournisseurs livrent presque tous les jours (voir la carte de chaleur). Ce sont les couples à plus fort potentiel d'économie par la consolidation.

Après optimisation, ce graphique permettra de mesurer le gain réel, couple par couple.

---

### Coût par livraison

In [ ]:
# ---------- Coût de transport par livraison (coût de chaque camion calculé plus haut) ----------
cout_livraisons = (camions.groupby(["supplier_id", "plant", "date"])
                          .agg(nb_camions=("camion", "size"), poids=("poids", lambda s: s.sum(min_count=1)),   # NaN = poids inconnu
                               distance=("distance", "first"), cout=("cout", "sum")))
mediane = cout_livraisons["cout"].median()

cout_livraisons["cout"].plot(kind="hist", bins=40, color="steelblue", edgecolor="black",
                             figsize=(8, 4), label="livraisons")
plt.axvline(mediane, color="black", linestyle="--", label=f"médiane : {mediane:,.2f} €")
plt.title(f"Coût de transport par livraison ({len(cout_livraisons)} livraisons, "
          f"total {cout_livraisons['cout'].sum():,.0f} €)".replace(",", " "))
plt.xlabel("Coût de la livraison (€)")
plt.ylabel("Nombre de livraisons")
plt.legend()
plt.grid(axis="y", alpha=0.5)
plt.show()

# Les livraisons les plus chères (barres isolées à droite de l'histogramme)
cout_livraisons.nlargest(5, "cout")

#### Analyse : coût par livraison

La plupart des livraisons coûtent peu (médiane : **72 €**), ce qui va avec des camions qui dépassent rarement 4 000 kg.

Les livraisons les plus chères sont maintenant celles qui contiennent **plusieurs lignes sans poids** : par exemple **SUP_00153 → PLT_004** les 10/11, 13/11 et 17/11, avec 3 camions réservés chacune (888 €). Leur coût vient uniquement de l'estimation haute des camions réservés.

Viennent ensuite **SUP_00046 → PLT_004** le 11/11 et le 02/12 (591,84 € chacune) : ce sont deux des 4 livraisons trop lourdes pour un seul camion (46,7 t et 32,3 t). Elles partent en 2 camions de plus de 4 t, à 91 km, soit 2 × 295,92 €. Leur coût vient du nombre de camions et non d'un mauvais remplissage : il y a peu à gagner sur ces livraisons.

L'optimisation vise à baisser le coût total en faisant moins de livraisons. Chaque livraison regroupera plus de besoins : le coût moyen par livraison devrait donc augmenter, alors que le coût total baisse.

---

## 5. Estimation des gains

La baseline donne le coût de la situation actuelle. Avant de construire le moteur de consolidation, on estime avec un algorithme simple ce que la consolidation peut rapporter : c'est l'objectif d'amélioration, chiffré, que le moteur devra viser.

### Principe : l'algorithme de base

On garde toutes les règles de la baseline : mêmes lignes de besoin (silver et quarantaine), mêmes camions réservés pour les lignes sans poids, même grille tarifaire, mêmes estimations hautes pour les données manquantes. On ne lève qu'une restriction : **un camion n'est plus limité aux besoins d'un seul jour**. Il peut emporter les besoins des jours suivants, livrés en avance (SF-06).

Pour chaque couple fournisseur × usine, les lignes sont triées par date de besoin, puis du plus lourd au plus léger :
1. un camion part le jour du **plus ancien besoin restant** ;
2. il prend les besoins jusqu'à sa date **+ 7 jours**, du plus ancien au plus récent, tant qu'il reste de la place (**24,5 t et 110,4 m³**, SF-07). Une ligne qui ne rentre pas reste pour un camion suivant, et les lignes plus légères qui la suivent peuvent encore compléter le camion ;
3. quand la fenêtre de 7 jours est dépassée ou que plus rien ne rentre, le camion est fermé, et le suivant part au plus ancien besoin restant.

```
Couple au planning de besoins quotidien
besoin restant le plus ancien : lundi 16/11    ──►  camion le lundi 16/11 (prend les besoins du 16/11 au 23/11)
besoin restant le plus ancien : mardi 24/11    ──►  camion le mardi 24/11 (prend les besoins du 24/11 au 01/12)
besoin restant le plus ancien : mercredi 02/12 ──►  camion le mercredi 02/12 ...
```

**Ce que l'estimation respecte** : les contraintes dures du projet. Aucun camion ne dépasse sa capacité, aucune livraison n'est en retard, aucune n'a plus de 7 jours d'avance, un camion ne sert qu'un couple, et chaque ligne est livrée une fois, entière.

**Ce qu'elle ignore** : les **jours fixes** (SF-04) et le **lissage** (SF-08). Chaque camion part le jour qui arrange son couple, sans regarder les autres couples de l'usine. Ces deux règles ajoutent des contraintes, donc du coût : l'estimation est une **estimation haute du gain**, l'objectif dont le moteur devra s'approcher.

Les **lignes sans poids** gardent leur camion réservé, le jour du besoin : sans leur poids, on ne peut pas les regrouper. Elles coûtent autant dans la baseline que dans l'estimation.

In [ ]:
# ---------- Estimation des gains : algorithme de base (fenêtre de 7 jours, sans jours fixes) ----------
FENETRE = pd.Timedelta(days=7)   # avance maximale d'une livraison sur son besoin (SF-06)


def eur(x):
    """177320.53 -> '177 321 €'."""
    return f"{x:,.0f} €".replace(",", " ")


def consolider_couple(lignes):
    """Camions d'un couple. Les lignes sont triées par date de besoin, puis du plus lourd au plus léger.
    Un camion part le jour du plus ancien besoin restant et prend les besoins des 7 jours suivants,
    tant qu'il reste de la place."""
    restantes = list(lignes.itertuples())
    camions_couple = []
    while restantes:
        date = restantes[0].need_date
        poids, volume, contenu, i = 0.0, 0.0, [], 0
        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date + FENETRE:
                break                                  # hors fenêtre : les suivantes aussi, camion fermé
            if poids + ligne.weight > POIDS_MAX or volume + ligne.volume > VOLUME_MAX:
                i += 1                                 # ne rentre pas : on essaie la ligne suivante
                continue
            poids, volume = poids + ligne.weight, volume + ligne.volume
            contenu.append(ligne.Index)
            restantes.pop(i)
            i = 0                                      # on repart de la plus ancienne ligne restante
        camions_couple.append({"date": date, "poids": poids, "volume": volume, "lignes": contenu})
    return camions_couple


def estimer(besoins):
    """Consolide chaque couple fournisseur × usine et chiffre ses camions.
    Retourne une table des camions et une table des lignes de besoin (avec leur camion et leur avance)."""
    besoins = besoins.sort_values(["plant", "supplier_id", "need_date", "weight"],
                                  ascending=[True, True, True, False])
    liste = [{"supplier_id": supplier, "plant": plant, **camion}
             for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"])
             for camion in consolider_couple(lignes_couple)]
    camions_est = pd.DataFrame(liste)
    camions_est["camion_id"] = range(1, len(camions_est) + 1)
    camions_est["nb_lignes"] = camions_est["lignes"].str.len()
    distance = (camions_est["supplier_id"].map(distances.set_index("supplier_id")["distance"])
                .fillna(DISTANCE_PAR_DEFAUT))         # fournisseur sans distance : > 650 km
    camions_est["cout"] = [cout_transport(p, d) for p, d in zip(camions_est["poids"], distance)]

    affectation = camions_est[["camion_id", "date", "lignes"]].explode("lignes")
    lignes_est = besoins.loc[affectation["lignes"]].assign(camion_id=affectation["camion_id"].values,
                                                           date_livraison=affectation["date"].values)
    lignes_est["avance_jours"] = (lignes_est["need_date"] - lignes_est["date_livraison"]).dt.days
    return camions_est.drop(columns="lignes"), lignes_est


# Lignes au poids connu (silver + quarantaine, la ligne de plus de 24,5 t reste exclue)
a_consolider = schedule[~schedule["reserve"]]
camions_est, lignes_est = estimer(a_consolider)

# Contrôles : les mêmes que pour la baseline, plus l'avance
assert (camions_est["poids"] <= POIDS_MAX).all() and (camions_est["volume"] <= VOLUME_MAX).all()  # SF-07
assert not lignes_est.index.duplicated().any()                       # SF-12 : aucune ligne livrée deux fois
assert set(lignes_est.index) == set(a_consolider.index)              # SF-12 : aucune ligne oubliée
assert abs(camions_est["poids"].sum() - a_consolider["weight"].sum()) < 1e-6   # aucun poids perdu
assert lignes_est["avance_jours"].between(0, FENETRE.days).all()     # jamais en retard, au plus 7 jours d'avance

# Les lignes sans poids gardent leur camion réservé, au même coût que dans la baseline
potentiel = pd.concat([camions_est.assign(reserve=False),
                       camions.loc[camions["reserve"], ["supplier_id", "plant", "date", "poids", "volume",
                                                        "nb_lignes", "cout", "reserve"]]],
                      ignore_index=True)


def resume(c):
    """Indicateurs d'un ensemble de camions (poids et remplissage : camions au poids connu)."""
    connus = c[~c["reserve"]]
    par_livraison = c.groupby(["supplier_id", "plant", "date"])["cout"].sum()
    return pd.Series({
        "Camions": nombre(len(c)),
        "dont réservés (poids inconnu)": nombre(c["reserve"].sum()),
        "Livraisons (couple × jour)": nombre(len(par_livraison)),
        "Coût total": eur(c["cout"].sum()),
        "Poids médian d'un camion": f"{nombre(round(connus['poids'].median()))} kg",
        "Remplissage médian en poids": pourcent(connus["poids"].median() / POIDS_MAX),
        "Camions de plus de 4 000 kg": pourcent((connus["poids"] > TRANCHES_KG[-1]).mean()),
        "Coût médian d'une livraison": eur(par_livraison.median()),
    })


cout_base, cout_est = camions["cout"].sum(), potentiel["cout"].sum()
silver_est, _ = estimer(schedule[schedule["source"] == "silver"])   # même calcul sans la quarantaine
cout_base_silver = camions.loc[camions["source"] == "silver", "cout"].sum()

print(f"Économie estimée : {eur(cout_base - cout_est)} sur les 6 semaines, soit {pourcent(1 - cout_est / cout_base)} "
      f"({eur((cout_base - cout_est) / camions['semaine'].nunique())} par semaine)")
print(f"Sans les lignes de la quarantaine : {eur(cout_base_silver)} -> {eur(silver_est['cout'].sum())}, "
      f"soit {pourcent(1 - silver_est['cout'].sum() / cout_base_silver)}")
print(f"Part des camions réservés dans le coût estimé : {pourcent(camions.loc[camions['reserve'], 'cout'].sum() / cout_est)}")
print(f"Avance moyenne d'une ligne : {nombre(lignes_est['avance_jours'].mean())} jours, "
      f"livrées le jour même : {pourcent((lignes_est['avance_jours'] == 0).mean())}, "
      f"avance max : {lignes_est['avance_jours'].max()} jours")
pd.DataFrame({"Baseline": resume(camions), "Estimation": resume(potentiel)})

#### Analyse : gain estimé

| | Baseline | Estimation | Évolution |
|---|---|---|---|
| Camions | 1 741 (dont 134 réservés) | **692** (558 consolidés + 134 réservés) | ÷ 2,5 |
| Livraisons (couple × jour) | 1 665 | 659 | ÷ 2,5 |
| Coût total | 177 321 € | **109 513 €** | **-67 807 € (-38,2 %)** |
| Poids médian d'un camion | 323 kg | 895 kg | × 2,8 |
| Remplissage médian en poids | 1,3 % | 3,7 % | |
| Camions de plus de 4 000 kg | 4,4 % | 18,1 % | |
| Coût médian d'une livraison | 72 € | 146 € | × 2 |

- **Le gain à viser** : regrouper les besoins de 7 jours au plus divise le nombre de camions par 2,5 et économise **67 807 € sur les 6 semaines, soit -38,2 %** (environ 11 300 € par semaine), sans aucune livraison en retard ni capacité dépassée.
- **Les données manquantes limitent le gain** : les 134 camions réservés (33 506 €) ne changent pas et pèsent **30,6 % du coût estimé**. Sur les seules lignes du silver, le gain monte à **-47,2 %** (142 965 € -> 75 461 €). Renseigner les poids manquants reste un levier à part entière.
- **Des camions plus lourds, mais encore loin d'être pleins** : le poids médian est multiplié par 2,8, mais seuls 18,1 % des camions dépassent 4 000 kg. La plupart des couples ne livrent pas assez, même sur 7 jours, pour remplir un camion : **l'économie vient du nombre de camions, pas de leur taille**.
- **Comme prévu en partie 4**, le coût d'une livraison double (72 € -> 146 € en médiane), puisqu'elle regroupe plus de besoins, alors que le coût total baisse.
- **Une avance modérée** : une ligne est livrée en moyenne 3,0 jours avant son besoin, 30,6 % le jour même, et jamais plus de 7 jours avant.

---

### Gains par usine et par semaine

In [ ]:
# ---------- Gains par usine et par semaine ----------
potentiel["semaine"] = potentiel["date"].dt.isocalendar().week
COMPARAISON = {"Baseline": camions, "Estimation": potentiel}
COULEURS = {"Baseline": "#9ec5d6", "Estimation": "royalblue"}

camions_sem = pd.DataFrame({nom: c.groupby("semaine").size() for nom, c in COMPARAISON.items()})
cout_sem = pd.DataFrame({nom: c.groupby("semaine")["cout"].sum() for nom, c in COMPARAISON.items()})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, table, titre, fmt in [(axes[0], camions_sem, "Camions par semaine", "%.0f"),
                              (axes[1], cout_sem, "Coût de transport par semaine (€)", "%.0f €")]:
    table.plot(kind="bar", ax=ax, color=[COULEURS[n] for n in table.columns], edgecolor="black", rot=0)
    for conteneur in ax.containers:
        ax.bar_label(conteneur, fmt=fmt, fontsize=8, padding=2)
    ax.set_xticks(range(len(table)), [f"S{s}" for s in table.index])
    ax.set_title(titre)
    ax.set_xlabel("Semaine")
    ax.margins(y=0.1)
    ax.grid(axis="y", alpha=0.4)
    ax.set_axisbelow(True)
fig.suptitle("Baseline et estimation des gains", fontsize=14)
plt.tight_layout()
plt.show()

# Camions par usine et par jour : la charge des quais
par_jour = {nom: c.groupby(["plant", "date"]).size() for nom, c in COMPARAISON.items()}
for nom, serie in par_jour.items():
    print(f"{nom} : {nombre(serie.groupby('date').sum().mean())} camions par jour ouvré en moyenne, "
          f"jusqu'à {serie.max()} camions un même jour pour une usine ({serie.idxmax()[0]}, {serie.idxmax()[1]:%d/%m})")
print("Camions de PLT_002 par semaine (estimation) :",
      {int(s): n for s, n in potentiel[potentiel["plant"] == "PLT_002"].groupby("semaine").size().items()})

# Tableau par usine
par_usine = pd.DataFrame({
    "Camions baseline": camions.groupby("plant").size(),
    "Camions estimation": potentiel.groupby("plant").size(),
    "Coût baseline": camions.groupby("plant")["cout"].sum(),
    "Coût estimation": potentiel.groupby("plant")["cout"].sum(),
})
par_usine.loc["Total"] = par_usine.sum()
par_usine["Gain"] = par_usine["Coût baseline"] - par_usine["Coût estimation"]
par_usine["Gain (%)"] = par_usine["Gain"] / par_usine["Coût baseline"]
par_usine["Part du gain"] = par_usine["Gain"] / par_usine.loc["Total", "Gain"]
par_usine.style.format({"Camions baseline": nombre, "Camions estimation": nombre, "Coût baseline": eur,
                        "Coût estimation": eur, "Gain": eur, "Gain (%)": pourcent, "Part du gain": pourcent})

#### Analyse : gains par usine et par semaine

| Usine | Camions | Coût | Gain | Part du gain |
|---|---|---|---|---|
| PLT_004 | 556 -> 193 | 58 588 € -> 33 314 € | **-43,1 %** | 37,3 % |
| PLT_001 | 540 -> 199 | 45 310 € -> 27 793 € | -38,7 % | 25,8 % |
| PLT_002 | 452 -> 197 | 45 491 € -> 28 996 € | -36,3 % | 24,3 % |
| PLT_003 | 193 -> 103 | 27 932 € -> 19 409 € | -30,5 % | 12,6 % |

- **PLT_004 confirme son statut de cible prioritaire** (parties 2 et 4) : c'est l'usine qui gagne le plus, en euros comme en pourcentage. Ses fournisseurs livrent presque tous les jours avec des camions presque vides : les regrouper divise ses camions par 2,9.
- **PLT_003 gagne le moins** (-30,5 %) : ses camions sont déjà les plus chargés (partie 3), et une bonne part de son coût vient des camions réservés de SUP_00129 (partie 4), que la consolidation ne touche pas.
- **Le pic de S48 disparaît** : PLT_002 passe de 163 camions à 47 cette semaine-là, autant qu'en S46 et S47. Livrer en avance étale le pic des 26 et 27/11 sur les jours précédents. Le coût de S48 passe de 39 899 € à 21 820 €, et la courbe des coûts devient régulière, puis baisse en décembre avec l'activité.
- **Les camions passent nettement sous les livraisons d'aujourd'hui**, comme attendu en partie 2 : environ 23 camions par jour ouvré au lieu de 58.
- **Mais l'estimation ne lisse pas la charge** : une usine reçoit encore jusqu'à 33 camions un même jour, car chaque couple livre le jour qui l'arrange. Le moteur de consolidation devra corriger ce point en appliquant les jours fixes (SF-04) et le lissage (SF-08), au prix d'une partie du gain.

---

### Classement des couples selon leur potentiel

Le **potentiel** d'un couple est son gain estimé : coût de la baseline moins coût de l'estimation, camions réservés compris (ils sont identiques dans les deux, et s'annulent donc dans le gain). On le met en regard du nombre de jours de livraison par semaine aujourd'hui, en moyenne sur les semaines où le couple est livré.

In [ ]:
# ---------- Classement des couples selon leur potentiel ----------
COUPLE = ["supplier_id", "plant"]
# Jours de livraison par semaine aujourd'hui (moyenne sur les semaines où le couple est livré)
frequence = livraisons.groupby(COUPLE + ["semaine"]).size().groupby(COUPLE).mean()

classement = pd.DataFrame({
    "Livraisons / sem. actuelles": frequence,
    "Coût baseline": camions.groupby(COUPLE)["cout"].sum(),
    "dont camions réservés": camions[camions["reserve"]].groupby(COUPLE)["cout"].sum(),
    "Coût estimation": potentiel.groupby(COUPLE)["cout"].sum(),
}).fillna({"dont camions réservés": 0})
classement["Gain"] = classement["Coût baseline"] - classement["Coût estimation"]
classement["Gain (%)"] = classement["Gain"] / classement["Coût baseline"]
classement = classement.sort_values("Gain", ascending=False)
classement["Part cumulée du gain"] = classement["Gain"].cumsum() / classement["Gain"].sum()
classement.insert(0, "Rang", range(1, len(classement) + 1))
classement.index = [f"{s} × {p}" for s, p in classement.index]

assert abs(classement["Gain"].sum() - (cout_base - cout_est)) < 1e-6   # le classement retrouve le gain total
gagnants = classement["Gain"] > 0.005
n_80 = int((classement["Part cumulée du gain"] < 0.8).sum()) + 1
print(f"{len(classement)} couples : {gagnants.sum()} gagnent à la consolidation, {(~gagnants).sum()} n'y gagnent rien "
      f"(livraisons / sem. actuelles médianes : {nombre(classement.loc[~gagnants, 'Livraisons / sem. actuelles'].median())})")
print(f"Les 10 premiers couples font {pourcent(classement['Part cumulée du gain'].iloc[9])} du gain, "
      f"et {n_80} couples ({pourcent(n_80 / len(classement))}) en font 80 %")

# ---------- Graphiques : les 15 couples au plus fort potentiel, et la concentration du gain ----------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.5), gridspec_kw={"width_ratios": [3, 2]})
top15 = classement.head(15).iloc[::-1]                       # le plus fort potentiel en haut
top15[["Coût baseline", "Coût estimation"]].plot(kind="barh", ax=ax1, edgecolor="black",
                                                 color=[COULEURS["Baseline"], COULEURS["Estimation"]])
ax1.legend(["Baseline", "Estimation"])
for conteneur in ax1.containers:
    ax1.bar_label(conteneur, fmt="%.0f €", fontsize=8, padding=2)
ax1.set_title("Les 15 couples au plus fort potentiel (coût sur les 6 semaines)")
ax1.set_xlabel("Coût (€)")
ax1.margins(x=0.12)
ax1.grid(axis="x", alpha=0.4)
ax1.set_axisbelow(True)

ax2.plot(classement["Rang"], classement["Part cumulée du gain"] * 100, color="royalblue", linewidth=2.5)
ax2.axhline(80, color="red", linestyle="--", label=f"80 % du gain : {n_80} couples")
ax2.axvline(n_80, color="red", linestyle="--")
ax2.set_title("Part cumulée du gain, couples classés par potentiel")
ax2.set_xlabel("Nombre de couples (du plus fort au plus faible potentiel)")
ax2.set_ylabel("Part du gain total (%)")
ax2.set_xlim(0, len(classement))
ax2.set_ylim(0, 105)
ax2.grid(alpha=0.4)
ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()

classement.head(15).style.format({"Livraisons / sem. actuelles": nombre, "Coût baseline": eur,
                                  "dont camions réservés": eur, "Coût estimation": eur, "Gain": eur,
                                  "Gain (%)": pourcent, "Part cumulée du gain": pourcent})

In [ ]:
# ---------- Potentiel selon la fréquence de livraison actuelle ----------
classes = pd.cut(classement["Livraisons / sem. actuelles"], bins=[0, 1, 2, 3, 5],
                 labels=["1 jour", "plus de 1 à 2 jours", "plus de 2 à 3 jours", "plus de 3 à 5 jours"])
par_frequence = classement.groupby(classes, observed=True).agg(
    Couples=("Gain", "size"), **{"Coût baseline": ("Coût baseline", "sum"), "Gain": ("Gain", "sum")})
par_frequence["Gain (%)"] = par_frequence["Gain"] / par_frequence["Coût baseline"]
par_frequence["Part du coût baseline"] = par_frequence["Coût baseline"] / par_frequence["Coût baseline"].sum()
par_frequence["Part du gain"] = par_frequence["Gain"] / par_frequence["Gain"].sum()
par_frequence.index.name = "Livraisons par semaine aujourd'hui"
par_frequence.style.format({"Coût baseline": eur, "Gain": eur, "Gain (%)": pourcent,
                            "Part du coût baseline": pourcent, "Part du gain": pourcent})

#### Analyse : couples à fort potentiel

**Un gain très concentré.** 141 couples sur 180 gagnent à la consolidation. Les 39 autres n'y gagnent rien : ils ne livrent déjà qu'une fois par semaine environ, et leurs besoins ne tombent jamais à moins de 7 jours d'écart. Les **10 premiers couples font 35,5 % du gain**, et **48 couples (26,7 %) en font 80 %**.

**Le potentiel suit la fréquence de livraison actuelle :**

| Livraisons par semaine aujourd'hui | Couples | Part du coût baseline | Gain | Part du gain |
|---|---|---|---|---|
| 1 jour | 58 | 9,8 % | -12,0 % | 3,1 % |
| plus de 1 à 2 jours | 70 | 25,7 % | -30,9 % | 20,8 % |
| plus de 2 à 3 jours | 20 | 14,2 % | -42,2 % | 15,7 % |
| plus de 3 à 5 jours | 32 | 50,3 % | **-46,0 %** | **60,5 %** |

Les 32 couples qui livrent plus de 3 jours par semaine pèsent la moitié du coût et 60,5 % du gain : ce sont eux que la carte de chaleur (partie 2) désignait. À l'inverse, les 58 couples qui livrent un seul jour par semaine ne rapportent presque rien.

**Les couples à fort potentiel** :
- **SUP_00085 → PLT_004** arrive en tête, comme attendu en partie 4 : 7 429 € -> 1 657 € (-78 %), soit 8,5 % du gain total à lui seul ;
- **7 des 15 premiers couples livrent PLT_004** (SUP_00085, SUP_00080, SUP_00145, SUP_00087, SUP_00149, SUP_00024, SUP_00011), ce qui confirme la priorité donnée à cette usine ;
- **SUP_00129 → PLT_003**, le couple le plus cher de la baseline, n'est que 10e : 10 144 € -> 8 481 € (-16 %). Ses 24 camions réservés (7 102 €) ne bougent pas. Pour ce couple, comme pour SUP_00129 → PLT_001, SUP_00001 → PLT_004 et SUP_00153 → PLT_004, le gain passe d'abord par la qualité des données.

---

## Conclusion

Sur les 6 semaines du planning, la baseline compte **1 665 livraisons, 1 741 camions et 177 321 € de transport**, lignes de la quarantaine comprises (1 599 livraisons, 1 603 camions et 142 965 € sur les seules lignes du silver). C'est la référence qui permettra de mesurer l'économie apportée par la consolidation. La ligne de plus de 24,5 t (SUP_00074, 26,9 t) est exclue en attendant la réponse du client.

**Ce que montrent les analyses**

1. **Une livraison = un camion.** Hors camions réservés, seules 4 livraisons, trop lourdes, en demandent deux. Le nombre de camions suit donc celui des livraisons : 290 par semaine en moyenne, soit près de 60 par jour ouvré.
2. **Des livraisons très fréquentes.** Les couples les plus actifs livrent chaque jour ouvré, presque toutes les semaines, et la plupart livrent PLT_004.
3. **Des camions presque vides.** Le poids médian est de 323 kg, soit 1,3 % de la capacité, et 95,6 % des camions pèsent moins de 4 000 kg. En volume, aucun camion ne dépasse 33,2 % de la capacité : c'est le poids, et non le volume, qui limitera la consolidation.
4. **Une grille tarifaire qui récompense le regroupement.** Le prix au kilo baisse avec le poids, et le prix d'un camion est quasi fixe au-delà de 4 000 kg. Or une livraison coûte 72 € en médiane : la baseline paie surtout beaucoup de petits envois, chers au kilo.
5. **Une charge irrégulière.** Le pic de S48 (402 camions, 39 899 €) vient surtout de PLT_002, puis l'activité baisse jusqu'à S51 (173 camions, 17 782 €). Dans la semaine, le jeudi est le jour le plus chargé et le mercredi le plus calme.
6. **Des données manquantes qui coûtent cher.** Les 134 lignes sans poids demandent chacune un camion réservé : **33 506 €, soit 18,9 % du coût**, en estimation haute. Cette part ne baissera pas avec la consolidation : seule une meilleure qualité des données (poids renseignés) peut la réduire.

**L'objectif d'amélioration**

Regrouper les besoins de 7 jours au plus dans les mêmes camions, sans retard ni capacité dépassée (algorithme de base, partie 5), ramène la baseline à **692 camions et 109 513 €** : une **économie estimée de 67 807 €, soit -38,2 %** sur les 6 semaines. Sur les seules lignes du silver, elle atteint -47,2 % : les camions réservés pèsent alors 30,6 % du coût restant.

Cette estimation ignore les jours fixes (SF-04) et le lissage (SF-08) : c'est une estimation haute, l'objectif dont le moteur de consolidation devra s'approcher en respectant ces deux règles. Plus il conserve de ce gain tout en les respectant, meilleur il est.

**Où chercher les économies**

- **Les couples qui livrent plus de 3 jours par semaine** : 32 couples, la moitié du coût et **60,5 % du gain estimé**. 48 couples font 80 % du gain, alors que 39 couples, qui livrent déjà environ une fois par semaine, n'y gagnent rien.
- **PLT_004 en priorité** : c'est l'usine la plus chère (58 588 €) et celle qui gagne le plus (-43,1 %, 37,3 % du gain). Sa charge est stable, ses fournisseurs livrent tous les jours et elle compte 7 des 15 couples au plus fort potentiel. Un profil fixe de 1 ou 2 livraisons par semaine (SF-04, SF-05) y sera valable toutes les semaines.
- **Les couples aux données complètes**, à commencer par SUP_00085 → PLT_004 (-5 772 €, -78 %, 8,5 % du gain à lui seul).
- **La qualité des données** : les couples chers à cause de leurs lignes sans poids (SUP_00129, SUP_00001, SUP_00153) gagnent peu. Renseigner les poids manquants permettrait de consolider ces lignes au lieu de leur réserver un camion. C'est un levier hors de notre main, à remonter au client.
- **PLT_002**, très irrégulière : la livraison en avance (SF-06) absorbe déjà le pic de S48 (163 -> 47 camions). Les jours fixes et le lissage (SF-08) devront préserver cet acquis.
- À l'inverse, les livraisons déjà lourdes, comme celles de SUP_00046 → PLT_004 (les plus chères, vers 600 €), ont des camions bien remplis, et PLT_003, aux camions les plus chargés, gagne le moins (-30,5 %).

**Ce qu'on attend après optimisation** : environ 2,5 fois moins de camions et un coût total en baisse d'un peu moins de 38 % (les jours fixes et le lissage coûteront une partie du gain estimé), avec une courbe des camions nettement sous celle des livraisons d'aujourd'hui. Les camions seront plus lourds (poids médian d'environ 900 kg), mais la plupart resteront sous 4 000 kg : **l'économie viendra du nombre de camions, pas de leur remplissage**. Le coût par livraison augmentera (146 € en médiane dans l'estimation), et le moteur devra répartir les livraisons dans la semaine (SF-08), ce que l'estimation ne fait pas encore (jusqu'à 33 camions un même jour pour une usine).